In [ ]:
import mne
import matplotlib
import matplotlib.pyplot as plt
import numpy as py
import pandas as pd
import os
import sys
from pathlib import Path
import numpy as np

import glob
import matplotlib
import pickle
import gc
from tqdm.notebook import tqdm

matplotlib.use('Qt5Agg')

## Paths

#### When running on lab computer

In [ ]:
PROJECT_DIR = Path.cwd().resolve()
while PROJECT_DIR != PROJECT_DIR.parent and not (PROJECT_DIR / "project_config.json").is_file():
    PROJECT_DIR = PROJECT_DIR.parent
if not (PROJECT_DIR / "project_config.json").is_file():
    raise FileNotFoundError("Could not find project_config.json above the notebook working directory")
sys.path.insert(0, str(PROJECT_DIR))
from project_config import get_path

PATH = get_path("project_root")
expt = 'IdvDiff'
subj = "R1234" # put subject name here
meg_dir = str(get_path("meg") / subj)
log_dir = str(get_path("logs"))
fig_dir = str(get_path("figures"))

os.chdir(PATH)
from helperfunctions import get_raws, find_white_rectangle_photodiode

In [ ]:
# create cache for pickled variables
cache_dir = os.path.join(meg_dir, 'cache')
if not os.path.exists(cache_dir):
    os.makedirs(cache_dir)

# create preprocessing directory
preproc_dir = meg_dir + '/preproc/'
if not os.path.exists(preproc_dir):
    os.makedirs(preproc_dir)

print(preproc_dir, cache_dir)

## Misc settings

In [ ]:
np.random.seed(123)

# analysis settings
photodiode_realignment = True
reject = {'mag': 3e-12}
interpolate = False

# Preprocessing

Look at the raw data to see if there's anything incredibly wrong with it

Normally here I would see if there are any bad channels etc; it's just good practice to look at what the raw data looks like! 
This is where we annotate any bad or noisy segments.

In [ ]:
raw = get_raws(meg_dir)
raw.plot(duration=60, n_channels = 157, highpass=0.5, filtorder=4)

### Filter the data

In [ ]:
raw_filt = raw.filter(1,40, method='iir') 
raw_filt.compute_psd(fmax=40).plot()

In [ ]:
events = mne.find_events(raw_filt, consecutive=False, min_duration = 0.002) # check that the triggers are equal to the number of total trials
print(events)

#### __FOR REVIEW ONLY__: line below checks the number of events by channel

In [ ]:
np.unique(events[:,2], return_counts=True)

## Compute ICA on raw object

In [ ]:
# make a copy object to def bad channels I want to exclude during ICA computation
ica_raw = raw_filt.copy()
ica_raw.info['bads']=['MEG 015','MEG 122','MEG 079'] # define bad channels that you've noticed in the psd plot here; they will only be removed to compute the ICA

In [ ]:
# fit the ica on the custom raw object
ICA = mne.preprocessing.ICA(n_components=0.95, random_state=123, verbose = True)
ica_fname = os.path.join(preproc_dir, f'{subj}_{expt}_ica-raw.fif')
ICA.fit(ica_raw,reject=reject, reject_by_annotation=True)

raw_filt.plot_sensors(show_names=True) # to see the sensor map

In [ ]:
ICA.plot_sources(ica_raw)
ICA.plot_components()

In [ ]:
ICA.plot_overlay(raw_filt,picks='mag') # need to close the components plot first

In [ ]:
raw_filt.plot_sensors(show_names=True) # to see the sensor map

In [ ]:
ICA_data = ICA.apply(raw_filt)
ICA_data.save(ica_fname, overwrite=True) 

# save the deleted components
pickle.dump(ICA.exclude, open(cache_dir + f'/ica_excluded.p', 'wb'))

## Remove and interpolate bad channels

#### __FOR REVIEW ONLY__: line below for checking the flagged channels

In [ ]:
# read in ICA raw file
ICA_data = mne.io.read_raw_fif(os.path.join(preproc_dir, f'{subj}_{expt}_ica-raw.fif'), preload=True)

with open(cache_dir + f'/bad_channels.p', 'rb') as f:
    saved_bads = pickle.load(f)
    print(saved_bads)

In [ ]:
ICA_data.info['bads'] = saved_bads

#### Preprocessing as usual

In [ ]:
# remove bad channels with the gui; recheck annotations
ICA_data.plot(n_channels = 157)

In [ ]:
bads = ICA_data.info['bads']

print(str(len(bads)) + ' bad channels')
print(ICA_data.info['bads']) # double check that the channels in bads are the actual bad channels
print(bads)

# save the bad channels to a temp variable
pickle.dump(bads, open(cache_dir + f'/bad_channels.p', 'wb'))

# no interpolation because the noise covariance needs to exclude the same channels
if interpolate:
    cleaned = ICA_data.interpolate_bads(reset_bads=True)
    print("CHANNELS HAVE NOT BEEN INTERPOLATED") if len(cleaned.info['ch_names']) != 193 else None
else:
    cleaned = ICA_data
    print("CHANNELS HAVE NOT BEEN INTERPOLATED")
 # make sure that all of the channels have been interpolated; check the number of chans we want

__FOR REVIEW ONLY:__ 

In [ ]:
cleaned = ICA_data

In [ ]:
print(bads)

# Epoch the data

In [ ]:
# Modified photodiode detection for white rectangle detection
print("Searching for white rectangle photodiode signal...")
best_channel, delay_info, response_percentage = find_white_rectangle_photodiode(ICA_data, trigger_channel='STI 014', max_delay_ms=150)

if best_channel:
    print(f"\nBest photodiode channel found: {best_channel}")
    print(f"Response rate: {response_percentage:.1f}%")
    print(f"Mean delay: {delay_info[0]:.1f} ± {delay_info[1]:.1f} ms")
    
    delay = delay_info[0]
else:
    print("No suitable photodiode channel found!")
    delay = 60

In [ ]:
events = mne.find_events(cleaned, consecutive=False, min_duration = 0.002) # check that the triggers are equal to the number of total trials
# will need to write a custom events object if analyzing p_exp data

info = cleaned.info  # get info object
picks_meg = mne.pick_types(info, meg=True, eeg=False, eog=False, stim=False) # select only the Meg channels

# Get the list of matching files
log_pattern = glob.glob(os.path.join(log_dir, subj, f'{subj}_logfile-group*.csv'))

logs = pd.read_csv(os.path.join(log_dir, subj, log_pattern[-1]))#, sep = "/t")

# use mean delay
if photodiode_realignment:
    print("Photodiode realignment")
    events[:,0] = events[:,0] + delay

# settings for epoch
event_id = {
    'scrambled_fast':160, 
    'scrambled_slow':161,
    'sents_fast':162,    
    'sents_slow':163,    
    'intact_fast':164,    
    'intact_slow':165,
    'tasks':166
}
epoch_tmin = -0.1
epoch_tmax = 0.8 # setting maximum epochs; will trim the data when I need to for analysis

print(events)

In [ ]:
# for subject R2614 only
new_events = np.delete(events,events[:,2]==78, axis = 0)

check the names of the channels that are causing epoch removal and maybe go back and delete them!

In [ ]:
# epoch the data!
epochs = mne.Epochs(
    cleaned,
    events=events,  # We use metadata instead of a static event dictionary
    tmin=epoch_tmin,  # 150ms before word onset
    tmax=epoch_tmax,  
    baseline=None,  # No baseline
    metadata=logs,  # Attach metadata to epochs
    preload=True,
    reject=reject, # reject epochs that have spikes higher than 3000ft
    picks=picks_meg
)

In [ ]:
# for participant 2549 and 2378 only!!! remove the duplicate trials
epochs = epochs['duplicate == False']
np.unique(epochs.events[:,2], return_counts=True) # check that the counts are sensible

In [ ]:
epochs.average().plot()

save the epochs

In [ ]:
epoched_fname = os.path.join(preproc_dir, f'{subj}_{expt}_final-epo.fif')
epochs.save(epoched_fname, overwrite=True)
print(epoched_fname)

#### __FOR REVIEW ONLY__: check the number of epochs dropped

In [ ]:
epoched_fname = os.path.join(preproc_dir, f'{subj}_{expt}_final-epo.fif')
epochs = mne.read_epochs(epoched_fname,preload=True)

In [ ]:
# line to check how many epochs were dropped
epochs.plot_drop_log()